# SOME/IP Anomaly Detection — Reproduction Pipeline

End-to-end pipeline reproducing and extending **Kim et al. (2026)**:
> Kim et al. *XGBoost-Based Anomaly Detection Framework for SOME/IP in In-Vehicle Networks.*  
> Systems, 14(2), 196. https://doi.org/10.3390/systems14020196

**Dataset**: 7 PCAP files from Figshare article 30970450  
https://figshare.com/articles/dataset/SOME_IP_traffic_normal_and_abnormal_traffic_/30970450

## Steps
| # | Script | Output | Tempo estimado |
|---|--------|--------|----------------|
| 0 | `00_download.py` | `data/raw/*.pcap` (~2 GB) | 5–30 min |
| 1 | `01_parse.py` | `data/parsed/parsed_packets.csv` (~3.5 GB) | 30–60 min |
| 2 | `03_features.py` | `data/features/*.npy` | 30–60 min |
| 3 | `04_train.py` | `results/v2_proposed/model_binary.json` | 2–5 min |
| 4 | `05_evaluate.py` | Métricas impressas | <1 min |

> Cada etapa verifica se o output já existe e pula automaticamente — re-execuções são seguras.

## Setup — local ou Google Colab

A célula abaixo detecta o ambiente automaticamente:
- **Colab**: monta o Drive, clona o repositório e instala dependências
- **Local**: usa o diretório atual

**No Colab**: os dados ficam em `Drive/someip_detection/` e persistem entre sessões.

In [ ]:
import subprocess, sys, os
from pathlib import Path

# ── Detecta Colab ─────────────────────────────────────────────────────────────
IN_COLAB = 'google.colab' in sys.modules or 'COLAB_RELEASE_TAG' in os.environ

if IN_COLAB:
    # Monta o Drive
    from google.colab import drive
    drive.mount('/content/drive')

    # Pasta de dados persistente no Drive
    DATA_DIR = Path('/content/drive/MyDrive/someip_detection')
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    os.environ['SOMEIP_DATA_DIR'] = str(DATA_DIR)

    # Clona o repositório (se ainda não existe)
    REPO_DIR = Path('/content/someip_detection')
    REPO_URL = 'https://github.com/SEU_USER/SEU_REPO'  # ← ajuste aqui
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=True)

    ROOT = REPO_DIR / 'detection'

    # Instala dependências
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    '-r', str(ROOT / 'requirements.txt')], check=True)
    print(f'Colab — dados em: {DATA_DIR}')

else:
    ROOT = Path().resolve().parent  # detection/ (rodando de notebooks/)
    print('Local')

SRC = ROOT / 'src'
sys.path.insert(0, str(ROOT))

print(f'ROOT = {ROOT}')
print(f'Python = {sys.version.split()[0]}')

def run(script, *args):
    """Executa um script src/ com output em tempo real."""
    cmd = [sys.executable, str(SRC / script), *args]
    print(f'\n▶  {" ".join(str(c) for c in cmd)}\n')
    result = subprocess.run(cmd, cwd=str(ROOT))
    if result.returncode != 0:
        raise RuntimeError(f'{script} falhou com código {result.returncode}')

## Step 0 — Download PCAPs

Baixa 7 PCAPs do Figshare via API pública (~2 GB total) para `data/raw/`.  
No Colab, os arquivos ficam no Drive e são reaproveitados em sessões futuras.

In [ ]:
run('00_download.py')

## Step 1 — Parse PCAPs → CSV

Lê os 7 PCAPs com Scapy e extrai campos IP, TCP/UDP e cabeçalho SOME/IP.  
Rotula cada pacote pelo IP de origem (atacantes conhecidos por PCAP).  
Saída: `data/parsed/parsed_packets.csv` (~14.2 M linhas, ~3.5 GB).

In [ ]:
run('01_parse.py')

## Step 2 — Extração de Features

Extrai 18 features (f01–f18) por pacote:

- **f01–f12**: Reprodução de Kim et al. (2026) — intervalos de tempo, likelihood/entropia de distribuição de bytes, distância Hamming entre payloads, variações de comprimento.
- **f13–f16**: Extensões — taxa de repetição, flag de source duplicada, comprimentos de payload SOME/IP e transporte.
- **f17**: `src_packet_rate` — pacotes/s deste src_ip em janela deslizante de 1000 pacotes.
- **f18**: `src_payload_diversity` — payloads únicos / total deste src_ip (mesma janela).

Saída: `data/features/{X,y}_{train,test}.npy`, `train_features.csv`, `test_features.csv`.

In [ ]:
run('03_features.py')

## Step 3 — Treinar XGBoost (binário)

Treina classificador binário XGBoost (normal vs ataque).  
Salva modelo em `results/v2_proposed/model_binary.json`.

> **Por que apenas binário?** O dataset rotula ataques no nível de PCAP, não de pacote —  
> tráfego de fundo nos PCAPs de ataque recebe o rótulo do tipo de ataque.  
> Classificação multi-classe confunde tráfego normal de fundo com classes de ataque.

In [ ]:
run('04_train.py', '--mode', 'binary')

## Step 4 — Avaliação

Carrega o modelo salvo e reporta recall por tipo de ataque e importância de features.

In [ ]:
run('05_evaluate.py', '--model', 'binary')

## Análise dos Resultados

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc,
    precision_score, recall_score, f1_score,
)
from paths import FEATURES_DIR, RESULTS_V2

FEAT_COLS = [
    'f01_ip_time_interval', 'f02_someip_likelihood', 'f03_someipsd_likelihood',
    'f04_tcpudp_likelihood', 'f05_someip_entropy', 'f06_someipsd_entropy',
    'f07_tcpudp_entropy', 'f08_someip_payload_changes', 'f09_someipsd_payload_changes',
    'f10_tcpudp_payload_changes', 'f11_ip_length_changes', 'f12_tcpudp_length_changes',
    'f13_payload_repeat_rate', 'f14_duplicate_source',
    'f15_someip_payload_length', 'f16_tcpudp_payload_length',
    'f17_src_packet_rate', 'f18_src_payload_diversity',
]

X_test  = np.load(FEATURES_DIR / 'X_test.npy')
y_test  = np.load(FEATURES_DIR / 'y_test.npy').astype(int)
at_test = np.concatenate([
    c['attack_type'].values
    for c in pd.read_csv(FEATURES_DIR / 'test_features.csv',
                          usecols=['attack_type'], chunksize=500_000)
])

model = xgb.XGBClassifier()
model.load_model(str(RESULTS_V2 / 'model_binary.json'))

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(f'F1        = {f1_score(y_test, y_pred):.4f}')
print(f'Precision = {precision_score(y_test, y_pred):.4f}')
print(f'Recall    = {recall_score(y_test, y_pred):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['Normal', 'Attack']).plot(
    ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix')

fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, lw=1.5, label=f'AUC = {roc_auc:.4f}')
axes[1].plot([0,1],[0,1],'--', color='grey')
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].set_title('ROC Curve'); axes[1].legend()

imps = model.feature_importances_
order = np.argsort(imps)[::-1]
top = 12
axes[2].barh([FEAT_COLS[i].split('_',1)[0] for i in order[:top]][::-1],
              imps[order[:top]][::-1])
axes[2].set_xlabel('Importance (gain)')
axes[2].set_title('Top-12 Features')

plt.tight_layout()
out_dir = ROOT / 'results' / 'v2_proposed'
out_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(out_dir / 'overview.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
attack_types = ['dos', 'fuzzy', 'mitm']
recalls = []
for at in attack_types:
    mask = (at_test == at)
    yt = y_test[mask]; yp = y_pred[mask]
    tp = ((yt==1) & (yp==1)).sum()
    fn = ((yt==1) & (yp==0)).sum()
    rc = tp / (tp+fn) if (tp+fn) > 0 else 0.0
    recalls.append(rc)
    print(f'{at:<8}  n={mask.sum():>9,}  recall={rc:.4f}')

fig, ax = plt.subplots(figsize=(5, 3))
bars = ax.bar(attack_types, recalls, color=['#e05c5c','#f0a060','#5c8ae0'])
ax.bar_label(bars, fmt='%.4f', padding=3)
ax.set_ylim(0, 1.08)
ax.set_ylabel('Recall')
ax.set_title('Recall por tipo de ataque (f01–f18)')
plt.tight_layout()
plt.savefig(out_dir / 'recall_by_type.png', dpi=150, bbox_inches='tight')
plt.show()

## Resultados

| Métrica | Kim et al. (12 features) | Este trabalho (18 features) |
|---------|--------------------------|-----------------------------|
| F1      | ~0.98 (reportado) / ~0.84 (reproduzido) | **~0.9979** |
| Recall DoS   | — | **~0.9998** |
| Recall Fuzzy | — / ~0.34 reproduzido | **~0.9991** |
| Recall MITM  | — | **~0.9972** |

### Nota sobre recall fuzzy
O dataset rotula **todos** os pacotes dos PCAPs fuzzy como "fuzzy", incluindo ~83% de
tráfego normal de fundo. A melhoria de recall vem de f17 (`src_packet_rate`), que
distingue o atacante (172.18.0.12/0.17) dos hosts normais pelo volume de tráfego.

### Paradoxo do f17
f17 aparece com apenas ~3.6% de importância por gain, mas removê-lo derruba o F1
de 0.9979 para 0.74. O XGBoost o usa uma vez perto da raiz como split quase perfeito
e delega o restante para outras features.